# 1. Agent vs LLM  

结构化输出是把模型结果约束为 JSON、Pydantic 对象、函数参数等可被程序直接消费的数据。LLM 更关注单次回答的格式，Agent 更关注多步骤任务的过程与最终结果。

## 1.1 核心差异

| 维度 | LLM 结构化输出 | Agent 结构化输出 |
| --- | --- | --- |
| 粒度 | 单次模型调用 | 多步骤任务流程 |
| 约束对象 | 最终回答格式 | 工具参数、中间状态、最终结果 |
| 典型能力 | schema 绑定、JSON 输出、函数调用 | 规划、工具调用、状态管理、重试修复 |
| 复杂度 | 低 | 高 |

## 1.2 控制方式

LLM 结构化输出主要通过 schema 控制回答形状，例如 `model.with_structured_output(schema)`。

Agent 结构化输出通常在执行工具、观察结果、必要时重试后，再把最终结果整理成指定 schema。

## 1.3 可靠性

| 维度 | LLM | Agent |
| --- | --- | --- |
| 格式稳定性 | schema 简单时较稳定 | 可通过校验和重试修复 |
| 事实准确性 | 依赖模型自身 | 可借助检索、数据库、API |
| 可观测性 | 主要看输入和输出 | 可查看每一步动作和工具结果 |

## 1.4 成本与复杂度

LLM 结构化输出通常一次调用完成，成本低、延迟低、实现简单。

Agent 可能包含多轮模型调用和多次工具调用，能力更强，但成本、延迟和调试难度更高。

## 1.5 适用场景

| 场景 | 推荐方式 |
| --- | --- |
| 文本抽取、分类、固定 JSON 生成 | LLM 结构化输出 |
| 查询外部系统后汇总 | Agent 结构化输出 |
| 多步骤排查、自动操作、工单创建 | Agent 结构化输出 |
| 简单字段补全 | LLM 结构化输出 |

## 1.6 选择原则

优先使用 LLM 结构化输出。只有当任务需要工具调用、多步骤决策、外部数据或失败恢复时，再引入 Agent。

# 2. 结构化输出的四种策略

In [ ]:
import os
import json
from  dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",  
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

# rprint(model.profile)

# 1. define tools
tavily = TavilySearch(
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
    max_results=5,
    search_depth="basic",
    topic="general",
    include_answer=True,
)

class SearchInput(BaseModel):
    query: str = Field(..., description="需要搜索验证的问题或关键词")

@tool(args_schema=SearchInput)
def web_search(query: str) -> str:
    """联网搜索并返回用于事实核验的结果。"""
    result = tavily.invoke({"query": query})
    return json.dumps(result, ensure_ascii=False)


# 2. Pydantic class
class Person(BaseModel):
    """ 个人基本信息."""
    name: str = Field(..., description="The person's name")
    gender: str = Field(..., description="The person's gender")
    age: int = Field(..., description="The person's age")

class Actor(BaseModel):
    """ 演员相关信息."""
    info: Person = Field(..., description="The personal information of the actor")
    role: str = Field(..., description="The role played by the actor in the movie")
    works: list[str] | None = Field(default=None, description="A list of other works the actor has been in")

class Movie(BaseModel):
    """ 电影相关信息."""
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The release year of the movie")
    rating: float  = Field(..., description="The movie's rating on a scale of 1 to 10")
    cast: list[Actor] = Field(..., description="A list of main cast members")
    story: str = Field(..., description="A brief summary of the movie's plot")


# 3. system prompt
system_prompt = """
你是一个专业智能助手。
回答任何涉及事实、时效或名单的问题前，必须先调用 web_search 工具进行搜索验证。
回答时只使用搜索结果中能验证的信息。
如果搜索结果不足以确认，请明确说明无法确认。
"""

## 2.1 ProviderStrategy  

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.messages import HumanMessage
from rich import print as rprint

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    response_format=ProviderStrategy(Movie)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="给出《肖申克的救赎》的详细信息")
    ]
})

rprint(response["structured_response"])

## 2.2 ToolStrategy

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    response_format=ToolStrategy(Movie)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="给出《肖申克的救赎》的详细信息")
    ]
})

rprint(response["structured_response"])

## 2.3 AutoStrategy

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.structured_output import AutoStrategy
from rich import print as rprint

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    response_format=AutoStrategy(Movie)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="给出《肖申克的救赎》的详细信息")
    ]
})

rprint(response["structured_response"])

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.structured_output import AutoStrategy, ToolStrategy, ProviderStrategy
from rich import print as rprint

model = init_chat_model(
    model="gpt-5.5",
    model_provider="openai",
    openai_api_key="sk-74a02e92ad8a4140858fd0ac2160092c",
    openai_api_base="https://sub2api.apizz.xyz/v1",
    default_headers={"User-Agent": "curl/8.0"},
)

# rprint(model.profile)

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    response_format=ProviderStrategy(Movie)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="给出《肖申克的救赎》的详细信息")
    ]
})

rprint(response["structured_response"])

# 3. ToolStrategy 详解

- ToolStrategy 是 LangChain **v1 的默认策略**：把 schema 伪装成一个 Tool，强制模型通过 tool calling 输出结构化数据。  

- 三种策略的关系： ResponseFormat = ToolStrategy | ProviderStrategy | AutoStrategy


- **构造参数：**

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| schema | Pydantic / dataclass / TypedDict / JSON Schema dict / Union | 必填 | 期望的输出结构，Union 会被自动展开为多个子 schema |
| tool_message_content | str \| None | None | 模型调用结构化输出 tool 后，返回的 ToolMessage 内容 |
| handle_errors | bool \| str \| Exception \| tuple \| Callable | True | 校验失败时的错误处理策略 |

In [ ]:
import os
from  dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",  
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

## 3.1 schema - Pydantic

**特点**：校验能力最强，支持字段类型、默认值、字段说明、嵌套结构和复杂约束。

**使用场景**：推荐作为默认选择，适合需要严格校验、结构清晰、后续要直接使用 Python 对象的任务。

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint

class ContactInfo(BaseModel):
    """ 用户联系方式."""
    name: str = Field(description="The person's name")
    email: str = Field(description="The person's email")
    phone: str = Field(description="The person's phone number")


agent = create_agent(
    model=model,
    system_prompt="你是一个专业智能助手，帮助用户解决各类问题",
    response_format=ToolStrategy(ContactInfo)
)

human_massage = "小明的邮箱是 xiaoming@icloud.com"

response = agent.invoke({
    "messages":HumanMessage(human_massage)
})

rprint(response['structured_response'])

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from rich import print as rprint


# 1. 工具 1 — 城市坐标查询
class CoordinateInput(BaseModel):
    city: str = Field(description="城市名称，例如 'Beijing', 'Shanghai'")

@tool(args_schema=CoordinateInput)
def get_coordinates(city: str) -> dict:
    """查询城市的经纬度坐标。"""
    coordinates = {"北京": (39.9, 116.4), "上海": (31.2, 121.5), "广州": (23.1, 113.3)}
    lat, lon = coordinates.get(city, (0.0, 0.0))
    return {"city": city, "latitude": lat, "longitude": lon}


# 2. 工具 2 — 根据坐标查天气（依赖工具 1 的输出）
class WeatherInput(BaseModel):
    latitude: float = Field(description="纬度")
    longitude: float = Field(description="经度")
    date: str = Field(description="查询日期，格式 YYYY-MM-DD")

@tool(args_schema=WeatherInput)
def get_weather_by_coords(latitude: float, longitude: float, date: str) -> dict:
    """根据经纬度坐标查询指定日期的天气状况。"""
    return {
        "date": date,
        "temperature": 22.5,
        "humidity": 65,
        "condition": "晴转多云",
        "wind": "北风 3级",
    }


# 3. 结构化输出模型
class Coordinates(BaseModel):
    """经纬度坐标。"""
    latitude: float = Field(description="纬度")
    longitude: float = Field(description="经度")

class WeatherDetail(BaseModel):
    """天气详情。"""
    temperature: float = Field(description="气温(℃)")
    humidity: int = Field(description="相对湿度(%)")
    condition: str = Field(description="天气状况描述")

class WeatherReport(BaseModel):
    """天气查询报告。"""
    city: str = Field(description="城市名称")
    location: Coordinates = Field(description="城市经纬度坐标")
    weather: WeatherDetail = Field(description="天气详情")
    date: str = Field(description="查询日期")
    suggestion: str = Field(description="根据天气状况给出的出行/穿衣建议")


# 4. 创建 Agent（两个工具串行调用）
agent = create_agent(
    model=model,
    tools=[get_coordinates, get_weather_by_coords],
    system_prompt=(
        "你是一个专业气象助手。回答天气问题时分两步："
        "1.先查询城市的经纬度；"
        "2.再用经纬度查询天气；"
        "3.最后根据结果给出出行/穿衣建议。"
    ),
    response_format=ToolStrategy(WeatherReport),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="查询北京 2026-07-05 的天气，给我一份详细报告")
    ]
})

rprint(response["structured_response"])
rprint(response)

## 3.2 schema - TypedDict

**特点**：以字典形态表达结构，类型清晰，运行时开销低；字段校验和描述能力较弱。

**使用场景**：适合已有代码以字典传递数据，或只需要轻量类型提示和固定字段结构的任务。

In [ ]:
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from rich import print as rprint


class ContactInfo(TypedDict):
    """用户联系方式。"""
    name: str
    email: str
    phone: str


agent = create_agent(
    model=model,
    system_prompt="你是一个专业智能助手，帮助用户解决各类问题",
    response_format=ToolStrategy(ContactInfo),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="小明的邮箱是 xiaoming@icloud.com")
    ]
})

rprint(response["structured_response"])

In [ ]:
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint


# 1. 自定义工具（仍用 Pydantic 定义 args_schema）
class CoordinateInput(BaseModel):
    city: str = Field(description="城市名称")

@tool(args_schema=CoordinateInput)
def get_coordinates(city: str) -> dict:
    """查询城市的经纬度坐标。"""
    coordinates = {"北京": (39.9, 116.4), "上海": (31.2, 121.5), "广州": (23.1, 113.3)}
    lat, lon = coordinates.get(city, (0.0, 0.0))
    return {"city": city, "latitude": lat, "longitude": lon}


class WeatherInput(BaseModel):
    latitude: float = Field(description="纬度")
    longitude: float = Field(description="经度")
    date: str = Field(description="查询日期，格式 YYYY-MM-DD")

@tool(args_schema=WeatherInput)
def get_weather_by_coords(latitude: float, longitude: float, date: str) -> dict:
    """根据经纬度坐标查询指定日期的天气状况。"""
    return {
        "date": date,
        "temperature": 22.5,
        "humidity": 65,
        "condition": "晴转多云",
        "wind": "北风 3级",
    }


# 2. TypedDict 结构化输出
class Coordinates(TypedDict):
    """经纬度坐标。"""
    latitude: float
    longitude: float

class WeatherDetail(TypedDict):
    """天气详情。"""
    temperature: float
    humidity: int
    condition: str

class WeatherReport(TypedDict):
    """天气查询报告。"""
    city: str
    location: Coordinates
    weather: WeatherDetail
    date: str
    suggestion: str


# 3. 创建 Agent
agent = create_agent(
    model=model,
    tools=[get_coordinates, get_weather_by_coords],
    system_prompt=(
        "你是一个专业气象助手。回答天气问题时分两步："
        "1.先查询城市的经纬度；"
        "2.再用经纬度查询天气；"
        "3.最后根据结果给出出行/穿衣建议。"
    ),
    response_format=ToolStrategy(WeatherReport),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="查询北京 2026-07-05 的天气，给我一份详细报告")
    ]
})

rprint(response["structured_response"])

## 3.3 schema - dataclass

**特点**：基于 Python 标准库，定义轻量，适合复用已有数据结构；校验能力弱于 Pydantic。

**使用场景**：适合结构简单、主要依赖类型标注、不需要复杂字段校验的输出任务。

In [ ]:
from dataclasses import dataclass, field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from rich import print as rprint


@dataclass
class ContactInfo:
    """用户联系方式。"""
    name: str = field(metadata={"description": "The person's name"})
    email: str = field(metadata={"description": "The person's email"})
    phone: str = field(metadata={"description": "The person's phone number"})


agent = create_agent(
    model=model,
    system_prompt="你是一个专业智能助手，帮助用户解决各类问题",
    response_format=ToolStrategy(ContactInfo),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="小明的邮箱是 xiaoming@icloud.com")
    ]
})

rprint(response["structured_response"])


In [ ]:
from dataclasses import dataclass, field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint


# 1. 自定义工具（工具参数仍用 Pydantic 定义 args_schema）
class CoordinateInput(BaseModel):
    city: str = Field(description="城市名称")

@tool(args_schema=CoordinateInput)
def get_coordinates(city: str) -> dict:
    """查询城市的经纬度坐标。"""
    coordinates = {"北京": (39.9, 116.4), "上海": (31.2, 121.5), "广州": (23.1, 113.3)}
    lat, lon = coordinates.get(city, (0.0, 0.0))
    return {"city": city, "latitude": lat, "longitude": lon}


class WeatherInput(BaseModel):
    latitude: float = Field(description="纬度")
    longitude: float = Field(description="经度")
    date: str = Field(description="查询日期，格式 YYYY-MM-DD")

@tool(args_schema=WeatherInput)
def get_weather_by_coords(latitude: float, longitude: float, date: str) -> dict:
    """根据经纬度坐标查询指定日期的天气状况。"""
    return {
        "date": date,
        "temperature": 22.5,
        "humidity": 65,
        "condition": "晴转多云",
        "wind": "北风 3级",
    }


# 2. dataclass 结构化输出
@dataclass
class Coordinates:
    """经纬度坐标。"""
    latitude: float = field(metadata={"description": "纬度"})
    longitude: float = field(metadata={"description": "经度"})

@dataclass
class WeatherDetail:
    """天气详情。"""
    temperature: float = field(metadata={"description": "气温(℃)"})
    humidity: int = field(metadata={"description": "相对湿度(%)"})
    condition: str = field(metadata={"description": "天气状况描述"})

@dataclass
class WeatherReport:
    """天气查询报告。"""
    city: str = field(metadata={"description": "城市名称"})
    location: Coordinates = field(metadata={"description": "城市经纬度坐标"})
    weather: WeatherDetail = field(metadata={"description": "天气详情"})
    date: str = field(metadata={"description": "查询日期"})
    suggestion: str = field(metadata={"description": "出行/穿衣建议"})


# 3. 创建 Agent
agent = create_agent(
    model=model,
    tools=[get_coordinates, get_weather_by_coords],
    system_prompt=(
        "你是一个专业气象助手。回答天气问题时分两步："
        "1.先查询城市的经纬度；"
        "2.再用经纬度查询天气；"
        "3.最后根据结果给出出行/穿衣建议。"
    ),
    response_format=ToolStrategy(WeatherReport),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="查询北京 2026-07-05 的天气，给我一份详细报告")
    ]
})

rprint(response["structured_response"])


## 3.4 schema - JSON Schema

**特点**：直接使用通用 JSON Schema 描述结构，语言无关，便于跨系统共享。

**使用场景**：适合前后端共用 schema、接口协议已有 JSON Schema，或不需要生成 Python 对象的任务。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from rich import print as rprint


# JSON Schema dict 定义
contact_schema = {
    "type": "object",
    "title": "ContactInfo",
    "description": "用户联系方式",
    "properties": {
        "name": {"type": "string", "description": "The person's name"},
        "email": {"type": "string", "description": "The person's email"},
        "phone": {"type": "string", "description": "The person's phone number"},
    },
    "required": ["name", "email", "phone"],
}


agent = create_agent(
    model=model,
    system_prompt="你是一个专业智能助手，帮助用户解决各类问题",
    response_format=ToolStrategy(contact_schema),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="小明的邮箱是 xiaoming@icloud.com")
    ]
})

rprint(response["structured_response"])

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint


# 1. 自定义工具（仍用 Pydantic 定义 args_schema）
class CoordinateInput(BaseModel):
    city: str = Field(description="城市名称")

@tool(args_schema=CoordinateInput)
def get_coordinates(city: str) -> dict:
    """查询城市的经纬度坐标。"""
    coordinates = {"北京": (39.9, 116.4), "上海": (31.2, 121.5), "广州": (23.1, 113.3)}
    lat, lon = coordinates.get(city, (0.0, 0.0))
    return {"city": city, "latitude": lat, "longitude": lon}


class WeatherInput(BaseModel):
    latitude: float = Field(description="纬度")
    longitude: float = Field(description="经度")
    date: str = Field(description="查询日期，格式 YYYY-MM-DD")

@tool(args_schema=WeatherInput)
def get_weather_by_coords(latitude: float, longitude: float, date: str) -> dict:
    """根据经纬度坐标查询指定日期的天气状况。"""
    return {
        "date": date,
        "temperature": 22.5,
        "humidity": 65,
        "condition": "晴转多云",
        "wind": "北风 3级",
    }


# 2. JSON Schema 结构化输出
weather_schema = {
    "type": "object",
    "title": "WeatherReport",
    "description": "天气查询报告",
    "properties": {
        "city": {"type": "string", "description": "城市名称"},
        "location": {
            "type": "object",
            "description": "城市经纬度坐标",
            "properties": {
                "latitude": {"type": "number", "description": "纬度"},
                "longitude": {"type": "number", "description": "经度"},
            },
            "required": ["latitude", "longitude"],
        },
        "weather": {
            "type": "object",
            "description": "天气详情",
            "properties": {
                "temperature": {"type": "number", "description": "气温(℃)"},
                "humidity": {"type": "integer", "description": "相对湿度(%)"},
                "condition": {"type": "string", "description": "天气状况描述"},
            },
            "required": ["temperature", "humidity", "condition"],
        },
        "date": {"type": "string", "description": "查询日期"},
        "suggestion": {"type": "string", "description": "出行/穿衣建议"},
    },
    "required": ["city", "location", "weather", "date", "suggestion"],
}


# 3. 创建 Agent
agent = create_agent(
    model=model,
    tools=[get_coordinates, get_weather_by_coords],
    system_prompt=(
        "你是一个专业气象助手。回答天气问题时分两步："
        "1.先查询城市的经纬度；"
        "2.再用经纬度查询天气；"
        "3.最后根据结果给出出行/穿衣建议。"
    ),
    response_format=ToolStrategy(weather_schema),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="查询北京 2026-07-05 的天气，给我一份详细报告")
    ]
})

rprint(response["structured_response"])

## 3.5 schema - Union

**特点**：允许一个 Agent 在多个候选结构中选择最匹配的输出类型，适合分支明显的任务。

**使用场景**：适合输入可能对应不同业务对象的场景，例如联系人、日程、工单、订单等混合信息抽取。

In [ ]:
from typing import Union
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from rich import print as rprint


class ContactInfo(BaseModel):
    """用户联系方式。"""
    name: str = Field(description="联系人姓名")
    email: str = Field(description="联系人邮箱")
    phone: str | None = Field(default=None, description="联系人电话，没有则为 None")


class CalendarEvent(BaseModel):
    """日程事件。"""
    title: str = Field(description="日程标题")
    date: str = Field(description="日期，格式 YYYY-MM-DD")
    time: str = Field(description="时间，格式 HH:MM")
    location: str | None = Field(default=None, description="地点，没有则为 None")


agent = create_agent(
    model=model,
    system_prompt="你是信息抽取助手。根据用户输入选择联系人或日程结构返回。",
    response_format=ToolStrategy(Union[ContactInfo, CalendarEvent]),
)

contact_response = agent.invoke({
    "messages": [
        HumanMessage(content="小明的邮箱是 xiaoming@icloud.com")
    ]
})

event_response = agent.invoke({
    "messages": [
        HumanMessage(content="明天上午 10 点在上海办公室开项目复盘会")
    ]
})

rprint(contact_response["structured_response"])
rprint(event_response["structured_response"])


## 3.6 tool_message_content

**作用**：自定义结构化输出工具调用后返回给模型的 ToolMessage 文本。只影响 Agent 内部消息，不改变 structured_response 的结构化结果。

**使用场景**：当默认 ToolMessage 过长、包含敏感字段，或希望给模型一个更短的确认信号时使用。

额外说明：
tool_message_content 只影响 ToolStrategy 为**结构化输出**那次工具调用生成的 ToolMessage.content
它不会改：
- 普通工具调用的返回内容
- structured_response 本身
- 模型最终回答内容
- 校验失败时用于重试的错误 ToolMessage，那部分主要由 handle_errors 控制
如果模型先生成了错误结构化输出然后重试，前面的错误消息不会用 tool_message_content；成功那次结构化输出的 ToolMessage.content 才会用它。

**案例：对比默认 ToolMessage 与自定义 ToolMessage**

In [ ]:
from langchain.messages import ToolMessage
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from rich import print as rprint


class ContactInfo(BaseModel):
    """用户联系方式。"""
    name: str = Field(description="联系人姓名")
    email: str = Field(description="联系人邮箱")
    phone: str | None = Field(default=None, description="联系人电话，没有则为 None")


# 1. 默认：ToolMessage 通常会包含结构化输出结果
default_agent = create_agent(
    model=model,
    system_prompt="你是信息抽取助手，负责提取联系人信息。",
    response_format=ToolStrategy(ContactInfo),
)

user_input = "小明的邮箱是 xiaoming@example.com，电话是 13800138000"

default_response = default_agent.invoke({
    "messages": [HumanMessage(content=user_input)]
})


# 2. tool_message_content 只返回短确认文本
custom_agent = create_agent(
    model=model,
    system_prompt="你是信息抽取助手，负责提取联系人信息。",
    response_format=ToolStrategy(
        schema = ContactInfo,
        tool_message_content="联系方式已保存。",
    ),
)

custom_response = custom_agent.invoke({
    "messages": [HumanMessage(content=user_input)]
})


# 只打印 tool_message
def get_tool_message(resp : dict):
    for msg in resp["messages"]:
        if isinstance(msg, ToolMessage):
            rprint(msg)

get_tool_message(default_response)
rprint(default_response["structured_response"])

get_tool_message(custom_response)
rprint(custom_response["structured_response"])


## 3.7 handle_errors


**作用**：控制 ToolStrategy 在结构化输出校验失败后的处理方式。默认 `handle_errors=True`，当模型输出不符合 schema、返回多个结构化输出，或字段类型/约束校验失败时，LangChain 会把错误作为 ToolMessage 反馈给模型，让模型按 schema 重新生成。

**使用场景**：适合需要提高结构化输出成功率、允许模型自我修正的抽取和生成任务。如果希望在生产链路中快速暴露错误，并交给外层重试、日志或告警处理，可以设为 `False`；如果希望统一提示模型如何修正，可以传入字符串；如果只想处理特定错误，可以传入异常类型、异常元组或自定义函数。

**使用方式**：

- `True`：默认值，捕获结构化输出相关错误，并使用默认错误提示让模型重试。
- `False`：不自动处理错误，校验失败时直接抛出异常。
- `str`：使用固定错误提示作为 ToolMessage，适合给模型明确的修正要求。
- `Exception` / `tuple[Exception, ...]`：只对指定异常类型启用重试，其他异常直接抛出。
- `Callable[[Exception], str]`：根据异常内容动态生成错误提示，适合区分不同失败原因。

## 3.8 handle_errors - True & False
- True：捕获结构化输出相关错误，并使用默认错误提示让模型重试
- False：直接抛异常


In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint
from typing import Union


class ContactInfo(BaseModel):
    """联系人信息。"""

    name: str = Field(description="联系人姓名")
    email: str = Field(description="联系人邮箱")
    phone: str | None = Field(default=None, description="联系人电话，没有则为 None")


class EventDetails(BaseModel):
    """活动详情"""

    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


# Union[ContactInfo, EventDetails] 输出结构必须 ContactInfo 和 EventDetails二选一，如果二者的字段 input 均包含
# LLM 会分别加载 ContactInfo 和 EventDetails 进行结构化输出，工具调用报错 -> ToolMessage.content :
# 'Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.\n Please fix your mistakes.'
# langchain 默认 handle_errors=True 允许模型按错误提示自动修正，最终返回正确结果

agent = create_agent(
    model=model,
    system_prompt="你是信息抽取助手。只返回可从用户输入中确认的信息。",
    response_format=ToolStrategy(
        schema=Union[ContactInfo, EventDetails],
        tool_message_content="数据提取完成",
        handle_errors=True,
    ),
)

human_massage = (
    "张三（电子邮箱：mailto:zhang3@atguigu.com）将参加于2026年7月15日举行的公司年会"
)

response = agent.invoke({"messages": [HumanMessage(content=human_massage)]})

rprint(response)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint
from typing import Union


class ContactInfo(BaseModel):
    """联系人信息。"""

    name: str = Field(description="联系人姓名")
    email: str = Field(description="联系人邮箱")
    phone: str | None = Field(default=None, description="联系人电话，没有则为 None")


class EventDetails(BaseModel):
    """活动详情"""

    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


# Union[ContactInfo, EventDetails] 输出结构必须 ContactInfo 和 EventDetails二选一，如果二者的字段 input 均包含
# LLM 会分别加载 ContactInfo 和 EventDetails 进行结构化输出，handle_errors=False 直接抛出异常:
# 工具调用报错 -> ToolMessage.content :
# 'Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.\n Please fix your mistakes.'

agent = create_agent(
    model=model,
    system_prompt="你是信息抽取助手。只返回可从用户输入中确认的信息。",
    response_format=ToolStrategy(
        schema=Union[ContactInfo, EventDetails],
        tool_message_content="数据提取完成",
        handle_errors=False,
    ),
)

human_massage = (
    "张三（电子邮箱：mailto:zhang3@atguigu.com）将参加于2026年7月15日举行的公司年会"
)

response = agent.invoke({"messages": [HumanMessage(content=human_massage)]})

rprint(response)

## 3.9 handle_errors - String
传入字符串，统一告诉模型如何修正缺失字段


In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint
from typing import Union


class ContactInfo(BaseModel):
    """联系人信息。"""

    name: str = Field(description="联系人姓名")
    email: str = Field(description="联系人邮箱")
    phone: str | None = Field(default=None, description="联系人电话")


class EventDetails(BaseModel):
    """活动详情"""

    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

# Union[ContactInfo, EventDetails] 输出结构必须 ContactInfo 和 EventDetails二选一，如果二者的字段 input 均包含
# LLM 会分别加载 ContactInfo 和 EventDetails 进行结构化输出：
# 工具调用报错 -> ToolMessage.content : "如果字段复杂，只按照 EventDetails 格式返回活动详情"
# 最终只返回 EventDetails 数据


agent = create_agent(
    model=model,
    system_prompt="你是信息抽取助手。 只返回可从用户输入中确认的信息。",
    response_format=ToolStrategy(
        schema=Union[ContactInfo, EventDetails],
        handle_errors="如果字段复杂，只按照 EventDetails 格式返回活动详情",
    ),
)

human_massage = (
    "张三（电子邮箱：mailto:zhang3@atguigu.com）将参加于2026年7月15日举行的公司年会"
)

response = agent.invoke({"messages": [HumanMessage(content=human_massage)]})

rprint(response)

## 3.10 handle_errors - tuple[exc, ...]
传入异常类型，只处理结构化输出校验错误


默认情况下，LangChain会处理结构化输出处理时抛出的两类异常：
- **MultipleStructuredOutputsError**：
多结构化输出错误，当返回的工具调用请求数量大于1时，抛出该异常；默认情况下LangChain会拦截该异常并提醒模型重试。
- **StructuredOutputValidationError**：
输出结构化验证错误，当输出格式不符合结构化要求时，抛出上述异常；默认情况下LangChain会拦截该异常并自动重试。

In [ ]:
from typing import Literal

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field


class ScoreResult(BaseModel):
    """评分结果。"""

    item: Literal["方案A", "方案B", "方案C"] = Field(..., description="评分方案")
    score: int = Field(ge=0, le=100)


class Person(BaseModel):
    """人物信息"""

    name: str = Field(description="人物姓名")


msg1 = "给方案A打120分"
msg2 = "给小明的方案B打80分"

In [46]:
from langchain.agents.structured_output import (
    MultipleStructuredOutputsError,
    ToolStrategy,
)

agent_1 = create_agent(
    model=model,
    system_prompt="你是评分助手，基于用户输入打分",
    response_format=ToolStrategy(
        schema=ScoreResult,
        tool_message_content="评分已完成",
        handle_errors=MultipleStructuredOutputsError,
    ),
)
# 此时只拦截 MultipleStructuredOutputsError ，出现 StructuredOutputValidationError
agent_1.invoke({"messages": [HumanMessage(content=msg1)]})

StructuredOutputValidationError: Failed to parse structured output for tool 'ScoreResult': Failed to parse data to ScoreResult: 1 validation error for ScoreResult
score
  Input should be less than or equal to 100 [type=less_than_equal, input_value=120, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal.

In [61]:
from langchain.agents.structured_output import (
    ToolStrategy,
    StructuredOutputValidationError,
)

agent_2 = create_agent(
    model=model,
    system_prompt="你是一名评分助手",
    response_format=ToolStrategy(
        schema=Union[ScoreResult, Person],
        handle_errors=StructuredOutputValidationError,
    ),
)
# 此时只拦截 StructuredOutputValidationError ，出现 MultipleStructuredOutputsError
agent_2.invoke({"messages": [HumanMessage(content=msg2)]})

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ScoreResult, Person) when only one is expected.

In [66]:
from langchain.agents.structured_output import (
    ToolStrategy,
    StructuredOutputValidationError,
)

agent_3 = create_agent(
    model=model,
    system_prompt="你是一名评分助手",
    response_format=ToolStrategy(
        schema=Union[ScoreResult, Person],
        handle_errors=(
            StructuredOutputValidationError,
            MultipleStructuredOutputsError,
        ),
    ),
)
# 拦截 StructuredOutputValidationError 和 MultipleStructuredOutputsError

resp1 = agent_3.invoke({"messages": [HumanMessage(content=msg1)]})
rprint(resp1["structured_response"])

resp2 = agent_3.invoke({"messages": [HumanMessage(content=msg2)]})
rprint(resp2["structured_response"])

ScoreResult(item='方案A', score=100)


ScoreResult(item='方案B', score=80)


## 3.11 handle_errors - Callable


In [68]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint


class ScoreResult(BaseModel):
    """评分结果。"""
    item: str = Field(description="评分对象")
    score: int = Field(ge=0, le=100, description="0 到 100 的整数分数")


def build_error_message(exc: Exception) -> str:
    if "less than or equal to 100" in str(exc):
        return "score 必须是 0 到 100 的整数，请根据用户原意重新给出合法分数。"
    return "请按 schema 修正结构化输出。"


agent = create_agent(
    model=model,
    system_prompt="你是评分助手。评分必须满足 schema 约束。",
    response_format=ToolStrategy(
        schema=ScoreResult,
        handle_errors=build_error_message,
    ),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="给方案 A 评 120 分")
    ]
})

rprint(response)


{
    'messages': [
        HumanMessage(
            content='给方案 A 评 120 分',
            additional_kwargs={},
            response_metadata={},
            id='379fca0a-dcac-4bf5-a32e-efefb7caa975'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 53,
                    'prompt_tokens': 341,
                    'total_tokens': 394,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 85
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-pro',
                'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
                'id': '918ca1cf-2a99-4aba-b635-59721982d80f',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f338a-df5a-7a91-b1fd-3030203c564b-0',
            tool_calls=[
                {
                    'name': 'ScoreResult',
                    'args': {'item': '方案 A', 'score': 120},
                    'id': 'call_00_qszt4MQjGIMqjjQDj4id6574',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 341,
                'output_tokens': 53,
                'total_tokens': 394,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='score 必须是 0 到 100 的整数，请根据用户原意重新给出合法分数。',
            name='ScoreResult',
            id='893b7353-c623-4714-86ba-178119c84b7f',
            tool_call_id='call_00_qszt4MQjGIMqjjQDj4id6574'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 53,
                    'prompt_tokens': 435,
                    'total_tokens': 488,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 51
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-pro',
                'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
                'id': '6daba7a1-248d-4abb-962b-a0a6071feab9',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f338a-e50d-7d72-9506-290644172407-0',
            tool_calls=[
                {
                    'name': 'ScoreResult',
                    'args': {'item': '方案 A', 'score': 100},
                    'id': 'call_00_kaoTN8KswpK1nTEbmWVW1261',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 435,
                'output_tokens': 53,
                'total_tokens': 488,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: item='方案 A' score[